# Advanced Techniques for Language Models

**Mini-Assignment 2**

---

António Cruz (140129), Bruno Santos (140586), Pedro Miranda (129268)


# 1. Introduction

This notebook is the source of the Mini-Assignment 2 report. It documents the alignment stage of the project: starting from the domain-adapted Mini-Assignment 1 model, this notebook adds supervised fine-tuning, then preference alignment via Direct Preference Optimization with reinforcement learning from AI feedback, and compares the three resulting model states on a fixed prompt set.

Mini-Assignment 1 (continued pretraining) is the previous report; the Final Project (system integration) is the next. This notebook assumes the reader has read or has access to the Mini-Assignment 1 report.


## 1.1 Pipeline overview

The full post-training pipeline this project walks through has four stages:

1. Pretraining. A base model is trained on general text by its authors. We start from SmolLM2-360M (open-source).
2. Continued pretraining (Mini-Assignment 1). The base is adapted to the job-postings domain by continued next-token training on raw IT job descriptions. The result is a text completer fluent in the domain.
3. Supervised fine-tuning (Mini-Assignment 2, this stage). The domain-adapted model is taught to follow recruiter instructions, by training on pairs of (query, structured job description).
4. Preference alignment (Mini-Assignment 2, this stage). The supervised model is further trained to prefer outputs that respect constraints, structure and inclusive language, using preferences ranked by a stronger model acting as judge (Gemma-4 in our setup).

Each stage contributes a distinct capability: pretraining gives general language, continued pretraining gives domain fluency, supervised fine-tuning gives instruction-following, and preference alignment shapes the quality of those instruction-following outputs.


# 2. Starting point: the Mini-Assignment 1 checkpoint

Mini-Assignment 1 produced two trained variants of SmolLM2-360M: a full fine-tuning checkpoint and a LoRA adapter. The Mini-Assignment 1 report selected the LoRA variant as the best in-domain fit and the one carried forward.

Before any further training, the LoRA adapter is merged into the base model. The result is a single self-contained model that is equivalent at inference time to (base + Mini-Assignment 1 LoRA), but with no PEFT plumbing. This keeps the rest of the pipeline (supervised fine-tuning, then DPO) simpler: each stage trains a fresh LoRA on top of one consolidated base, with no stacked-adapter bookkeeping.


## 2.1 Merge the Mini-Assignment 1 LoRA into the base

The merge is a one-off operation: load the base model, attach the Mini-Assignment 1 LoRA, call `merge_and_unload`, and save the result. It runs on CPU and takes a few seconds. The base model id is read from the adapter config rather than hardcoded, so the cell stays robust to future model swaps.


In [1]:
# Merge the Mini-Assignment 1 LoRA into the SmolLM2-360M base.
# This runs on CPU: from_pretrained loads the weights to CPU and no tensors
# are moved to the GPU, so the GPU stays free for the training cells below.
# The merged model is saved to outputs/mp1-360m/merged/ and is the starting
# point for the supervised fine-tuning and DPO stages below.
import json
import time
import gc
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

# Locate the project root: walk up to a folder containing data/.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

LORA_DIR   = PROJECT_ROOT / "outputs" / "mp1-360m" / "lora"
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

# Read the base model id from the adapter config; never hardcoded.
adapter_cfg = json.loads((LORA_DIR / "adapter_config.json").read_text())
BASE_MODEL = adapter_cfg["base_model_name_or_path"]
print(f"Base model:    {BASE_MODEL}")
print(f"LoRA adapter:  {LORA_DIR}")
print(f"Merged output: {MERGED_DIR}")

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(model, str(LORA_DIR), torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(MERGED_DIR)
tokenizer.save_pretrained(MERGED_DIR)

# Free CPU memory before the rest of the notebook loads the model again.
del model, tokenizer
gc.collect()

print(f"Merged in {time.time() - t0:.1f}s.")


Base model:    HuggingFaceTB/SmolLM2-360M
LoRA adapter:  /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/lora
Merged output: /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged


[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged in 3.0s.


## 2.2 Load the merged model

With the merged checkpoint on disk, the rest of the notebook loads the model from `outputs/mp1-360m/merged/`. From here onwards, the model is a regular pretrained-style checkpoint and downstream stages do not need to know about the Mini-Assignment 1 LoRA at all.


In [2]:
# Load the merged Mini-Assignment 1 model for the rest of the notebook.
import torch
from pathlib import Path
from transformers import AutoModelForCausalLM, AutoTokenizer

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
MERGED_DIR = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"

ma1_tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1_model     = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
total_params  = sum(p.numel() for p in ma1_model.parameters())

print(f"Loaded merged MA1 model from {MERGED_DIR}")
print(f"Total parameters: {total_params:,}")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loaded merged MA1 model from /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged
Total parameters: 361,821,120


# 3. Supervised fine-tuning

A continued-pretraining checkpoint is a text completer, not an instruction-follower. Direct Preference Optimization and other alignment techniques assume the starting point already follows instructions, so a short supervised fine-tuning pass on (query, response) pairs is needed before alignment.

This section trains the merged Mini-Assignment 1 model on roughly 7,500 (recruiter query, structured job description) pairs produced by the atlm_teacher ETL agent. The result is a domain-adapted, instruction-following model ready for alignment.


## 3.1 Prompt template

Mini-Assignment 1 left the model as a domain-fluent text completer that has never seen instructions. Before any alignment can happen, the model must be taught to recognise where a prompt ends and where a response should begin, and what kind of response is expected. A consistent prompt template is what carries that signal during supervised fine-tuning.

The template adopted here is Alpaca-inspired, with a one-sentence system framing and two clearly delimited fields:

```
You are a recruitment assistant. Given a brief recruiter request, write a complete structured job posting in Markdown.

### Request
{query}

### Posting
{jd}
```

The same template is used at inference time: the prompt fed to the model is everything up to and including `### Posting\n`, and the model produces the job posting from there.

Three reasons for this choice over the alternatives considered (a leaner key-value template, and a ChatML chat-template setup):

1. Explicit task framing. SmolLM2-360M is small and has never been instruction-tuned. A one-sentence system preamble gives the model an unambiguous anchor for the task, which is worth its roughly 25-token cost.

2. No clash with response content. The structured job descriptions in the training data already use `##` Markdown headings (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`). The template uses `###` for its own separators, so the model can tell a template marker from a response heading without ambiguity. ChatML-style special tokens would not have this clash, but at the cost of growing the tokenizer's vocabulary.

3. No special-token machinery. Plain-text separators tokenise into existing vocabulary, so no new embeddings need to be learned and no chat template needs to be defined on the tokenizer. The HuggingFace TRL `SFTTrainer` consumes this through a simple formatting function, with no extra configuration.

The supervised fine-tuning stage below uses this template for every one of the roughly 7,500 training pairs.


## 3.2 Training data

The supervised fine-tuning data are the 2,507 records in `data/processed/converted.jsonl`, the output of the atlm_teacher ETL agent that took raw Djinni postings and produced clean, structured job descriptions in Markdown.

Each record has three independent recruiter queries pointing to the same job description. The three queries are fanned out into independent training examples, so the model sees the same target response paired with three different phrasings of the same underlying request. This is a cheap form of augmentation that teaches the model to be robust to phrasing variation, and it roughly triples the effective dataset size.

The records are split 90/10 into train and validation at the record level (not at the query level), so no job description appears in both splits. With a fixed seed of 42, the split is reproducible.


In [3]:
# Load the SFT data, fan out the three queries per record, format with the
# Section 3.1 template, and split into train/validation at the record level.
import json
import random
from pathlib import Path

from datasets import Dataset

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
SEED = 42

# Read raw records (each has `queries` (list of 3) and `job_description`).
records = [json.loads(ln) for ln in open(SFT_DATA, encoding="utf-8")]
print(f"Loaded {len(records):,} records from {SFT_DATA.name}")

# Split at the record level (not the query level), so no JD appears in both
# train and val. 90/10 split, seed 42.
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records   = records[:n_val]
train_records = records[n_val:]
print(f"Split: {len(train_records):,} train records, {len(val_records):,} val records")

# Prompt template, exactly as described in Section 3.1.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def format_example(query: str, jd: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n{jd}"
    )

def fan_out(rs):
    out = []
    for r in rs:
        for q in r["queries"]:
            out.append({"text": format_example(q, r["job_description"])})
    return out

train_examples = fan_out(train_records)
val_examples   = fan_out(val_records)
print(f"After fan-out: {len(train_examples):,} train examples, "
      f"{len(val_examples):,} val examples")

sft_train = Dataset.from_list(train_examples)
sft_val   = Dataset.from_list(val_examples)

# A quick look at one formatted training example, truncated for readability.
print()
print("Sample formatted example:")
print("-" * 60)
print(train_examples[0]["text"][:800])
print("...")


Loaded 2,507 records from converted.jsonl
Split: 2,257 train records, 250 val records
After fan-out: 6,771 train examples, 750 val examples

Sample formatted example:
------------------------------------------------------------
You are a recruitment assistant. Given a brief recruiter request, write a complete structured job posting in Markdown.

### Request
We need someone to help us ensure our product is high quality before it gets to the customer.

### Posting
# Manual Test Engineer

## Summary
We are looking for a motivated and technically skilled Test Engineer to join an international team. The primary focus of this role is ensuring the successful delivery of a high-quality product through rigorous testing and framework development.

## Required Skills
- Test case writing and execution
- Test framework development and maintenance
- Integration with CI/CD infrastructure
- Problem-solving skills
- Technical aptitude

## Responsibilities
- Write and execute tests on a daily basis to e

## 3.3 Training configuration

Supervised fine-tuning continues the parameter-efficient approach from Mini-Assignment 1: a fresh LoRA adapter is trained on top of the merged base, leaving the base weights untouched. This keeps both the disk footprint and the GPU memory cost small, and matches the LoRA shape (r=16, alpha=32, dropout=0.05, attention plus feed-forward projections) found to work well in the previous stage.

Hyperparameters mirror Mini-Assignment 1 where applicable: bf16 precision, micro-batch 4 with gradient accumulation 4 (effective batch 16), warmup ratio 0.03, weight decay 0.01, and seed 42. The maximum sequence length is 1024 tokens, which fits every (query, posting) pair without truncation (the longest examples are well under 800 tokens). Training runs for 2 epochs over the roughly 6,800 training examples; with the fanned-out queries each unique job description is seen six times in total (three phrasings, two epochs).

The learning rate is 2e-4, the conventional value for LoRA. The loss is computed over the entire formatted sequence (preamble plus request plus response) rather than only the response; for a small model this is a minor inefficiency, not a correctness issue, and keeps the configuration simple.

The trained adapter, log history and a summary file are written to `outputs/ma2-360m-sft/`. The run tag is `ma2-360m-sft`.


In [4]:
# Training configuration for the SFT run; consumed by run_sft() below.
SFT_RUN = "ma2-360m-sft"
SFT_OUT = PROJECT_ROOT / "outputs" / SFT_RUN

SFT_CFG = {
    "epochs": 2,
    "per_device_batch": 4,
    "grad_accum": 4,                   # effective batch = 16
    "learning_rate": 2e-4,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_seq_length": 1024,
    # LoRA shape, same as Mini-Assignment 1.
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "seed": SEED,
}
print("SFT config:")
for k, v in SFT_CFG.items():
    print(f"  {k}: {v}")
print(f"output dir: {SFT_OUT}")


SFT config:
  epochs: 2
  per_device_batch: 4
  grad_accum: 4
  learning_rate: 0.0002
  warmup_ratio: 0.03
  weight_decay: 0.01
  max_seq_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  seed: 42
output dir: /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft


## 3.4 Running supervised fine-tuning

The `run_sft` function below wraps the merged Mini-Assignment 1 model in a fresh LoRA, configures the HuggingFace TRL `SFTTrainer`, runs training, and saves the resulting adapter together with a small summary file (trainable parameter count, wall-clock minutes, final validation loss and perplexity). It mirrors the shape of the `run_training` function used in Mini-Assignment 1.

Running the next cell requires a CUDA GPU with bf16 support. On the RTX 4090 used here, the run takes roughly 10 to 15 minutes.


In [5]:
# Defines run_sft(): one supervised fine-tuning run on top of the merged
# base, producing a new LoRA adapter at outputs/ma2-360m-sft/.
import json
import math
import time

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig
from trl import SFTConfig, SFTTrainer

def run_sft():
    set_seed(SFT_CFG["seed"])
    SFT_OUT.mkdir(parents=True, exist_ok=True)

    # Load the merged Mini-Assignment 1 model (the SFT starting point).
    tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        MERGED_DIR, torch_dtype=torch.bfloat16
    )

    peft_cfg = LoraConfig(
        r=SFT_CFG["lora_r"],
        lora_alpha=SFT_CFG["lora_alpha"],
        lora_dropout=SFT_CFG["lora_dropout"],
        target_modules=SFT_CFG["lora_target_modules"],
        task_type="CAUSAL_LM",
    )

    args = SFTConfig(
        output_dir=str(SFT_OUT),
        num_train_epochs=SFT_CFG["epochs"],
        per_device_train_batch_size=SFT_CFG["per_device_batch"],
        per_device_eval_batch_size=SFT_CFG["per_device_batch"],
        gradient_accumulation_steps=SFT_CFG["grad_accum"],
        learning_rate=SFT_CFG["learning_rate"],
        warmup_ratio=SFT_CFG["warmup_ratio"],
        weight_decay=SFT_CFG["weight_decay"],
        max_length=SFT_CFG["max_seq_length"],
        bf16=True,
        eval_strategy="steps",
        eval_steps=100,
        logging_steps=20,
        save_strategy="no",
        seed=SFT_CFG["seed"],
        report_to=[],
        dataset_text_field="text",
    )

    trainer = SFTTrainer(
        model=model,
        args=args,
        train_dataset=sft_train,
        eval_dataset=sft_val,
        peft_config=peft_cfg,
        processing_class=tokenizer,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    trainer.save_model(str(SFT_OUT))
    tokenizer.save_pretrained(SFT_OUT)

    final = trainer.evaluate()
    summary = {
        "stage": "sft",
        "run": SFT_RUN,
        "learning_rate": SFT_CFG["learning_rate"],
        "trainable_params": trainable,
        "total_params": total,
        "epochs": SFT_CFG["epochs"],
        "train_examples": len(sft_train),
        "val_examples": len(sft_val),
        "minutes": round(minutes, 1),
        "final_eval_loss": round(final["eval_loss"], 4),
        "final_eval_perplexity": round(math.exp(final["eval_loss"]), 2),
    }
    (SFT_OUT / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1)
    )
    (SFT_OUT / "summary.json").write_text(json.dumps(summary, indent=1))
    print("done:", summary)
    return summary


In [6]:
# Runs supervised fine-tuning. Needs a CUDA GPU; about 10-15 minutes on the
# RTX 4090 used here.
summary_sft = run_sft()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/6771 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/6771 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/750 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
100,1.615488,1.605422,1.627626,521808.000000,0.631096
200,1.561771,1.554631,1.565704,1045638.000000,0.638605
300,1.540243,1.532224,1.528136,1562484.000000,0.642742
400,1.514234,1.520558,1.497237,2083092.000000,0.644680
500,1.467698,1.511047,1.492061,2599214.000000,0.646317
600,1.477231,1.503955,1.471643,3119204.000000,0.647349
700,1.467161,1.499548,1.472973,3640152.000000,0.648179
800,1.445180,1.496453,1.448786,4159128.000000,0.648546
848,1.462105,1.495557,1.464512,4405448.000000,0.648675


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Mean Token Accuracy
1.462105,1.495557,848,1.464512,4405448.000000,0.648675


done: {'stage': 'sft', 'run': 'ma2-360m-sft', 'learning_rate': 0.0002, 'trainable_params': 8683520, 'total_params': 370504640, 'epochs': 2, 'train_examples': 6771, 'val_examples': 750, 'minutes': 19.3, 'final_eval_loss': 1.4956, 'final_eval_perplexity': 4.46}


## 3.5 Sanity check: sample generations

A small set of prompts is run through both the merged Mini-Assignment 1 model (the starting point of this stage) and the supervised fine-tuned model. Side by side, the comparison should show that the SFT model:

1. Recognises the request as an instruction to produce a complete, structured job posting, rather than continuing the input as free text.
2. Produces output with the expected Markdown structure (`## Summary`, `## Required Skills`, `## Responsibilities`, `## Requirements`).
3. Stays on topic with respect to the recruiter query.

This is a qualitative check, not a metric. The full three-way comparison (base, Mini-Assignment 1 plus SFT, aligned) is in Section 6.


In [7]:
# Quick sanity check: a few prompts through MA1-only (merged) and MA1+SFT,
# greedy decoding for reproducibility.
from peft import PeftModel

"""
SANITY_PROMPTS = [
    "We need a senior data engineer who can design and operate batch and streaming pipelines on AWS.",
    "Looking for a UX designer to lead a small product team and ship a customer-facing dashboard.",
    "We are hiring a DevOps engineer comfortable with Kubernetes, CI/CD, and incident response.",
]
"""

SANITY_PROMPTS = [
    "We are looking for a Senior Data Scientist capable of helping junior members",
    "We need to reforce our team with a DevSecOps Team Lead",
    "Looking for a Functional Tester that can also work with business requirements",
]

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

def generate(model, tokenizer, prompt: str, max_new_tokens: int = 400) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True
    )

# Load MA1-merged (the starting point) and MA1+SFT.
ma1_tok   = AutoTokenizer.from_pretrained(MERGED_DIR)
ma1       = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
).to("cuda")
sft_base  = AutoModelForCausalLM.from_pretrained(
    MERGED_DIR, torch_dtype=torch.bfloat16
)
sft_model = PeftModel.from_pretrained(sft_base, str(SFT_OUT)).to("cuda")

for q in SANITY_PROMPTS:
    p = build_prompt(q)
    print("=" * 80)
    print("REQUEST:", q)
    print("=" * 80)
    print("\n--- merged MA1 (text completer, no SFT) ---")
    print(generate(ma1, ma1_tok, p))
    print("\n--- MA1 + SFT ---")
    print(generate(sft_model, ma1_tok, p))
    print()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

REQUEST: We are looking for a Senior Data Scientist capable of helping junior members

--- merged MA1 (text completer, no SFT) ---
We are looking for a Senior Data Scientist capable of helping junior members

### Responsibilities
- Develop and maintain data science pipelines
- Develop and maintain data science models
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate with other team members to ensure data quality and accuracy
- Collaborate w

# 4. Preference dataset construction

Direct Preference Optimization needs a dataset of (prompt, chosen response, rejected response) triples. Rather than manual annotation or a generic preference dataset, this project builds its own via reinforcement learning from AI feedback: candidates are sampled from the supervised-fine-tuned model and ranked by the atlm_teacher Gemma-4 agent acting as judge.

This section produces the preference dataset that the next section uses for DPO training.


## 4.1 Preference prompts

The prompts used to build the preference dataset are drawn from the supervised fine-tuning validation records (the 250 records held out at the 90/10 split in Section 3.2). The model never trained on these, so the candidates sampled in Section 4.2 reflect generalisation rather than memorised completions, which is what we want the preference signal to shape.

To keep the prompt set diverse, one query is picked per validation record. The records carry three phrasings of the same posting, but a single phrasing per posting is enough; using all three would give the judge near-duplicate prompts and waste judging calls. That yields 250 distinct recruiter prompts, written to `data/processed/ma2/preference_prompts.jsonl`. A defensive check confirms the set does not overlap with the 20 evaluation prompts from Section 6.1, which is guaranteed by construction (the evaluation prompts are hand-written, not part of `converted.jsonl`) but worth verifying.


In [8]:
# Section 4.1 - select preference prompts from the SFT validation split.
# CPU only. Reproducible with seed 42.
import json
import random
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA     = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
MA2_DIR      = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_FILE    = MA2_DIR / "eval_prompts.jsonl"
PREF_PROMPTS = MA2_DIR / "preference_prompts.jsonl"
SEED = 42

# Reproduce the Section 3.2 record-level split (seed 42, 90/10) so the val
# records here are exactly the records the SFT model never trained on.
records = [json.loads(l) for l in open(SFT_DATA, encoding="utf-8")]
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records = records[:n_val]
print(f"SFT val records (held out from training): {len(val_records):,}")

# One query per record, picked deterministically with a separate seed so
# the query pick does not correlate with the record shuffle.
rng = random.Random(SEED + 1)
selected = []
for r in val_records:
    q_idx = rng.randrange(len(r["queries"]))
    selected.append({
        "prompt_id": f"pref-{r['id'].replace(':', '-')}",
        "query": r["queries"][q_idx],
        "source_record_id": r["id"],
        "source": r.get("source", "unknown"),
    })
print(f"Selected preference prompts: {len(selected):,}")

# Defensive check: no overlap with the evaluation prompt set.
eval_queries = {json.loads(l)["query"] for l in open(EVAL_FILE, encoding="utf-8")}
overlap = sum(1 for p in selected if p["query"] in eval_queries)
print(f"Overlap with eval prompts (expected 0): {overlap}")

# Persist.
MA2_DIR.mkdir(parents=True, exist_ok=True)
with open(PREF_PROMPTS, "w", encoding="utf-8") as f:
    for p in selected:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"Wrote {PREF_PROMPTS.relative_to(PROJECT_ROOT)}")


SFT val records (held out from training): 250
Selected preference prompts: 250
Overlap with eval prompts (expected 0): 0
Wrote data/processed/ma2/preference_prompts.jsonl


## 4.2 Sampling candidates from the SFT model

For each of the 250 preference prompts, four candidate postings are sampled from the supervised fine-tuned model. Sampling (temperature 0.9, top-p 0.95) is used rather than greedy decoding because the candidates have to differ enough that the judge can rank them; greedy decoding would give four near-identical samples and produce no preference signal.

Setting `num_return_sequences=4` in a single `generate` call returns the four candidates in one batched forward pass, which is the cheap way to do this on a small model. The maximum new-token budget is 500, comfortably long enough for a structured Markdown posting.

The output is written to `data/processed/ma2/sft_candidates.jsonl`, one line per (prompt, candidate) pair. The cell is idempotent: prompts that already have four candidates on disk are skipped, so the run is resumable. After sampling, the GPU is freed (the SFT model is unloaded and the CUDA cache is cleared) so the judging step in Section 4.3 can use the GPU for Gemma-4 on `agent_server`.

This cell requires a CUDA GPU. On the RTX 4090 used here, 250 prompts at four candidates each take roughly 30 to 45 minutes.


In [9]:
# Section 4.2 - define sample_candidates(): sample k completions per
# preference prompt from the SFT model. Function is GPU-bound and is
# called by the next cell.
import gc
import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MERGED_DIR      = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"
SFT_OUT         = PROJECT_ROOT / "outputs" / "ma2-360m-sft"
MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
SEED = 42

# Same template as Section 3.1, defined here so Section 4 can run after a
# kernel restart without re-running Section 3.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

SAMPLING_CFG = {
    "k": 4,
    "max_new_tokens": 500,
    "temperature": 0.9,
    "top_p": 0.95,
    "seed": SEED,
}

def _done_prompt_ids(path, k):
    if not path.exists():
        return set()
    counts = {}
    for line in open(path, encoding="utf-8"):
        pid = json.loads(line)["prompt_id"]
        counts[pid] = counts.get(pid, 0) + 1
    return {pid for pid, n in counts.items() if n >= k}

def sample_candidates():
    set_seed(SAMPLING_CFG["seed"])

    # Load merged MA1 base + the SFT LoRA adapter on top, in bf16, on cuda.
    tok = AutoTokenizer.from_pretrained(MERGED_DIR)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    base  = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
    model = PeftModel.from_pretrained(base, str(SFT_OUT)).to("cuda").eval()

    prompts = [json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8")]
    done = _done_prompt_ids(CANDIDATES_FILE, SAMPLING_CFG["k"])
    todo = [p for p in prompts if p["prompt_id"] not in done]
    print(f"prompts total {len(prompts)} | already done {len(done)} | todo {len(todo)}")

    f = open(CANDIDATES_FILE, "a", encoding="utf-8")
    t0 = time.time()
    for i, p in enumerate(todo, 1):
        text = build_prompt(p["query"])
        enc = tok(text, return_tensors="pt").to(model.device)
        with torch.no_grad():
            out = model.generate(
                **enc,
                max_new_tokens=SAMPLING_CFG["max_new_tokens"],
                do_sample=True,
                temperature=SAMPLING_CFG["temperature"],
                top_p=SAMPLING_CFG["top_p"],
                num_return_sequences=SAMPLING_CFG["k"],
                pad_token_id=tok.eos_token_id,
            )
        in_len = enc["input_ids"].shape[1]
        for j, o in enumerate(out, 1):
            txt = tok.decode(o[in_len:], skip_special_tokens=True)
            f.write(json.dumps({"prompt_id": p["prompt_id"],
                                "candidate_idx": j,
                                "text": txt},
                               ensure_ascii=False) + "\n")
        f.flush()
        if i % 10 == 0 or i == len(todo):
            el = time.time() - t0
            print(f"  {i}/{len(todo)} prompts | {el/60:.1f} min | "
                  f"{i/max(el,1e-6):.2f} prompts/s")
    f.close()

    # Free the GPU so agent_server can use it for Gemma-4 in Section 4.3.
    del model, base, tok
    gc.collect()
    torch.cuda.empty_cache()
    print(f"done in {(time.time()-t0)/60:.1f} min, GPU freed")


In [10]:
# Sample candidates from the SFT model. GPU-bound; roughly 30-45 min
# on the 4090. Idempotent: re-running picks up where it left off.
sample_candidates()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

prompts total 250 | already done 250 | todo 0
done in 0.0 min, GPU freed


## 4.3 Judging candidates with Gemma-4

For each prompt, the four candidates are sent to the underlying Gemma-4 model on `agent_server` with a custom system prompt that contains the judging rubric. The judge is asked for a listwise ranking, returning the best and worst candidate in one call, because one call per prompt is much cheaper than the six pairwise calls that all-pairs comparison would need, and DPO needs only one (chosen, rejected) pair per prompt.

The rubric prioritises four criteria in order: faithfulness to the recruiter request, presence and completeness of the four required Markdown sections, professional and inclusive language, and absence of repetition or truncation. The judge is told to end its response with a single parseable line, `RANKING: best=<id> worst=<id>`, which the parser extracts.

Note on model selection: the call goes to the raw model id `gemma-4-e4b-it-q4-kxl-gguf` rather than to the `atlm_teacher` agent name. The `atlm_teacher` agent carries a system prompt for the ETL task (posting to queries + JD), not for judging; calling the model directly lets us substitute the judging rubric as the system prompt. The underlying LLM is still the same Gemma-4 used at the SFT data-preparation stage, so the self-preference and judge-equals-training-signal concerns noted in Section 7 apply.

The output is `data/processed/ma2/judge_ranks.jsonl`, one line per ranked prompt, capturing the best id, the worst id, and the raw judge response (kept for traceability). The step is idempotent: prompts already in the file are skipped. The agent_server `gemma-4` model has four inference slots, so the judge call uses four worker threads.

This cell requires `agent_server` to be running on `http://localhost:7701`.


In [11]:
# Section 4.3 - define judge_all(): for each preference prompt, listwise-
# rank its four candidates via Gemma-4 on agent_server, write best/worst.
import json
import queue
import re
import threading
import time
import urllib.error
import urllib.request
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
JUDGE_RANKS     = MA2_DIR / "judge_ranks.jsonl"

AGENT_SERVER  = "http://localhost:7701"
JUDGE_MODEL   = "gemma-4-e2b-it-q4-kxl-gguf"
ENDPOINT      = f"{AGENT_SERVER}/v1/chat/completions"
TIMEOUT       = 300
JUDGE_WORKERS = 4

JUDGE_SYSTEM = (
    "You are an expert recruiter and a careful writing reviewer.\n\n"
    "You will be shown a recruiter request and four candidate job postings, "
    "numbered 1 to 4. Your task is to rank the four candidates by overall "
    "quality.\n\n"
    "Apply the following criteria, in order of importance:\n\n"
    "1. Faithfulness to the request: the posting must reflect the role, "
    "technology stack, seniority and any constraints expressed in the "
    "request.\n"
    "2. Structure and completeness: the posting must include the four "
    "required sections '## Summary', '## Required Skills', "
    "'## Responsibilities' and '## Requirements', each with substantive, "
    "non-empty content.\n"
    "3. Professional and inclusive language: clear, neutral, free of biased "
    "or discriminatory phrasing.\n"
    "4. Absence of repetition, rambling, or truncation: each section must "
    "read cleanly without filler or mid-sentence stops.\n\n"
    "You may briefly note the strengths and weaknesses of each candidate "
    "before deciding.\n\n"
    "End your response with the single line:\n\n"
    "RANKING: best=<id> worst=<id>\n\n"
    "where <id> is the candidate number (1, 2, 3 or 4). Do not write "
    "anything after that line."
)

def _build_judge_user(query, candidates):
    parts = [f"RECRUITER REQUEST:\n{query}\n"]
    for i, c in enumerate(candidates, 1):
        parts.append(f"\n---\n\nCANDIDATE {i}:\n{c}\n")
    return "".join(parts)

def _call_judge(query, candidates):
    payload = {
        "model": JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": _build_judge_user(query, candidates)},
        ],
        "temperature": 0.0,
        "max_tokens": 16384,    # Gemma 4 has a 128K context; let it EOS naturally rather than cap arbitrarily
    }
    req = urllib.request.Request(
        ENDPOINT, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        resp = json.load(r)
    return resp["choices"][0]["message"]["content"]

_RANKING_RE = re.compile(r"RANKING:\s*best=(\d)\s+worst=(\d)", re.I)

def _parse_ranking(text):
    m = _RANKING_RE.search(text or "")
    if not m:
        return None
    best, worst = int(m.group(1)), int(m.group(2))
    if best == worst or best not in (1, 2, 3, 4) or worst not in (1, 2, 3, 4):
        return None
    return best, worst

def _done_judge_prompt_ids(path):
    if not path.exists():
        return set()
    return {json.loads(l)["prompt_id"] for l in open(path, encoding="utf-8")}

def judge_all():
    # Build prompt_id -> {candidate_idx: text} from sft_candidates.jsonl.
    by_prompt = {}
    for l in open(CANDIDATES_FILE, encoding="utf-8"):
        r = json.loads(l)
        by_prompt.setdefault(r["prompt_id"], {})[r["candidate_idx"]] = r["text"]

    prompts = [json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8")]
    done = _done_judge_prompt_ids(JUDGE_RANKS)
    todo = [p for p in prompts
            if p["prompt_id"] not in done
            and len(by_prompt.get(p["prompt_id"], {})) == 4]
    print(f"prompts with 4 candidates: {sum(1 for v in by_prompt.values() if len(v)==4)} "
          f"| already judged: {len(done)} | todo: {len(todo)}")
    if not todo:
        print("nothing to do.")
        return

    q = queue.Queue()
    for p in todo:
        q.put(p)
    f = open(JUDGE_RANKS, "a", encoding="utf-8")
    lock = threading.Lock()
    stats = {"ok": 0, "parse_fail": 0, "err": 0}
    t0 = time.time()

    def worker():
        while True:
            try:
                p = q.get_nowait()
            except queue.Empty:
                return
            cands = by_prompt[p["prompt_id"]]
            cand_list = [cands[i] for i in (1, 2, 3, 4)]
            try:
                raw = _call_judge(p["query"], cand_list)
            except Exception:
                with lock:
                    stats["err"] += 1
                continue
            parsed = _parse_ranking(raw)
            with lock:
                if parsed is None:
                    stats["parse_fail"] += 1
                else:
                    best, worst = parsed
                    f.write(json.dumps({
                        "prompt_id": p["prompt_id"],
                        "best": best, "worst": worst,
                        "raw": raw,
                    }, ensure_ascii=False) + "\n")
                    f.flush()
                    stats["ok"] += 1
                n_seen = stats["ok"] + stats["parse_fail"] + stats["err"]
                if n_seen % 20 == 0:
                    el = time.time() - t0
                    print(f"  ok={stats['ok']} parse_fail={stats['parse_fail']} "
                          f"err={stats['err']} | {el/60:.1f} min")

    threads = [threading.Thread(target=worker, daemon=True)
               for _ in range(JUDGE_WORKERS)]
    for t in threads: t.start()
    for t in threads: t.join()
    f.close()
    el = time.time() - t0
    print(f"done in {el/60:.1f} min | ok={stats['ok']} "
          f"parse_fail={stats['parse_fail']} err={stats['err']}")


In [12]:
# Judge all prompts. Requires agent_server up on http://localhost:7701.
# Idempotent: re-running picks up where it left off.
# Expected ~1-2 hours for 250 prompts at concurrency 4.
judge_all()


prompts with 4 candidates: 250 | already judged: 250 | todo: 0
nothing to do.


## 4.4 Final preference dataset

The DPO trainer in Section 5 expects a dataset of `(prompt, chosen, rejected)` strings. This cell joins the three intermediate files (the preference prompts, the SFT candidates, and the judge ranks) into that final shape:

- `prompt` is the formatted inference prompt for the recruiter query, ending with the template's `### Posting` marker, identical to the string the SFT model continued from when sampling, so DPO sees exactly the prompt the model is asked to complete.
- `chosen` is the best candidate's text for that prompt, as ranked by the judge.
- `rejected` is the worst.

The output, `data/processed/ma2/preferences.jsonl`, is the input to Section 5's `DPOTrainer`.


In [13]:
# Section 4.4 - assemble the final (prompt, chosen, rejected) preference
# dataset for DPO. CPU only.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR         = PROJECT_ROOT / "data" / "processed" / "ma2"
PREF_PROMPTS    = MA2_DIR / "preference_prompts.jsonl"
CANDIDATES_FILE = MA2_DIR / "sft_candidates.jsonl"
JUDGE_RANKS     = MA2_DIR / "judge_ranks.jsonl"
PREFERENCES     = MA2_DIR / "preferences.jsonl"

# Same template as Section 3.1, defined here for self-containment.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

# Build lookups.
prompts = {p["prompt_id"]: p for p in
           (json.loads(l) for l in open(PREF_PROMPTS, encoding="utf-8"))}
cands = {}
for l in open(CANDIDATES_FILE, encoding="utf-8"):
    r = json.loads(l)
    cands.setdefault(r["prompt_id"], {})[r["candidate_idx"]] = r["text"]
ranks = [json.loads(l) for l in open(JUDGE_RANKS, encoding="utf-8")]

n_in = len(ranks)
n_out = 0
with open(PREFERENCES, "w", encoding="utf-8") as f:
    for r in ranks:
        pid = r["prompt_id"]
        if pid not in prompts or pid not in cands or len(cands[pid]) != 4:
            continue
        f.write(json.dumps({
            "prompt":   build_prompt(prompts[pid]["query"]),
            "chosen":   cands[pid][r["best"]],
            "rejected": cands[pid][r["worst"]],
        }, ensure_ascii=False) + "\n")
        n_out += 1
print(f"Wrote {n_out:,}/{n_in:,} preference triples to "
      f"{PREFERENCES.relative_to(PROJECT_ROOT)}")


Wrote 250/250 preference triples to data/processed/ma2/preferences.jsonl


# 5. DPO training

Direct Preference Optimization aligns the supervised fine-tuned model so that, on the same prompt, it puts higher log-probability on the chosen response than on the rejected one. Compared with PPO it needs no separate reward model and no rollout loop, so it fits a single-GPU budget. We use HuggingFace TRL's `DPOTrainer`.

The starting point of this stage is the supervised fine-tuned model from Section 3. Following the same "merge between stages" pattern used at the MA1 to SFT boundary, the SFT adapter is first merged into the base to produce a single consolidated checkpoint at `outputs/ma2-360m-sft-merged/`, and a fresh LoRA is then trained on top of that for DPO. Two DPO runs are launched back to back with different beta (KL coefficient) values, 0.1 and 0.3, all other hyperparameters held constant. The brief weights an explicit discussion of beta and learning-rate sensitivity, so the two-beta sweep gives Section 6 a real data point on how aggressively the policy should diverge from the reference.


## 5.1 Merge the SFT adapter into the base

The SFT model (the merged MA1 base plus the Section 3 LoRA adapter) is consolidated into a single self-contained checkpoint by calling `merge_and_unload`. This mirrors the MA1 to SFT boundary and keeps the DPO stage simple: a fresh LoRA on a plain base, with no stacked-adapter bookkeeping.

This is a CPU operation, idempotent, and writes `outputs/ma2-360m-sft-merged/`. The result is the policy and the (implicit) reference for the DPO stage below.


In [14]:
# Section 5.1 - merge the SFT LoRA into the merged MA1 base. CPU only.
import json
import gc
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MERGED_DIR     = PROJECT_ROOT / "outputs" / "mp1-360m" / "merged"
SFT_OUT        = PROJECT_ROOT / "outputs" / "ma2-360m-sft"
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"

t0 = time.time()
print(f"Base (merged MA1): {MERGED_DIR}")
print(f"SFT adapter      : {SFT_OUT}")
print(f"Output (SFT merged): {SFT_MERGED_DIR}")

tokenizer = AutoTokenizer.from_pretrained(MERGED_DIR)
base  = AutoModelForCausalLM.from_pretrained(MERGED_DIR, torch_dtype=torch.bfloat16)
model = PeftModel.from_pretrained(base, str(SFT_OUT), torch_dtype=torch.bfloat16)
model = model.merge_and_unload()

SFT_MERGED_DIR.mkdir(parents=True, exist_ok=True)
model.save_pretrained(SFT_MERGED_DIR)
tokenizer.save_pretrained(SFT_MERGED_DIR)

(SFT_MERGED_DIR / "README.md").write_text(
    "# MA1+SFT merged into a single checkpoint\n\n"
    "This folder is the Mini-Assignment 1 LoRA (already merged in `outputs/mp1-360m/merged/`) "
    "plus the supervised fine-tuning LoRA from Section 3, both baked into the base weights, "
    "in bf16. Inference here is equivalent to loading the merged MA1 base and the SFT "
    "adapter together, with no PEFT plumbing.\n\n"
    "Provenance:\n"
    "- Base (merged MA1): `outputs/mp1-360m/merged/`\n"
    "- SFT adapter      : `outputs/ma2-360m-sft/`\n"
    "- Built by: notebook Section 5.1\n\n"
    "Purpose: starting point for the DPO stage in Section 5.\n"
)

del model, base, tokenizer
gc.collect()
print(f"Merged in {time.time() - t0:.1f}s.")


Base (merged MA1): /home/logus/env/iscte/atlm_pro/outputs/mp1-360m/merged
SFT adapter      : /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft
Output (SFT merged): /home/logus/env/iscte/atlm_pro/outputs/ma2-360m-sft-merged


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged in 2.8s.


## 5.2 Preference dataset

The `preferences.jsonl` file produced in Section 4.4 holds `(prompt, chosen, rejected)` triples. It is split 90/10 at the triple level into a DPO training set and an evaluation set, with seed 42. The evaluation set is used by the trainer to report a held-out preference metric (the fraction of pairs where the policy log-probability margin agrees with the judge's choice), which is the natural automatic metric for DPO and goes in Section 6.


In [15]:
# Section 5.2 - load preferences.jsonl and split 90/10 for DPO training.
import json
import random
from pathlib import Path

from datasets import Dataset

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

PREFERENCES = PROJECT_ROOT / "data" / "processed" / "ma2" / "preferences.jsonl"
SEED = 42

triples = [json.loads(l) for l in open(PREFERENCES, encoding="utf-8")]
print(f"Loaded {len(triples):,} preference triples")

random.Random(SEED).shuffle(triples)
n_val = max(1, len(triples) // 10)
val_triples   = triples[:n_val]
train_triples = triples[n_val:]
print(f"Split: {len(train_triples):,} train / {len(val_triples):,} eval")

dpo_train = Dataset.from_list(train_triples)
dpo_eval  = Dataset.from_list(val_triples)

# Quick check of the columns DPOTrainer expects.
print(f"Columns: {dpo_train.column_names}")


Loaded 250 preference triples
Split: 225 train / 25 eval
Columns: ['prompt', 'chosen', 'rejected']


## 5.3 Training configuration

The configuration mirrors Mini-Assignment 1 and the SFT stage where applicable, with the values specific to DPO set conservatively:

- Beta (KL coefficient): two values are run, 0.1 (the TRL and DPO-paper default) and 0.3 (more aggressive divergence from the reference). All other hyperparameters held constant across the two runs.
- Learning rate 5e-6, an order of magnitude lower than the SFT learning rate of 2e-4. DPO is nudging the policy, not retraining it; higher learning rates collapse the policy fast.
- One epoch over the roughly 225 training triples, at effective batch 8 (micro-batch 2 with gradient accumulation 4). Halved from SFT's effective batch of 16 because DPO forwards through both chosen and rejected each step, raising memory pressure.
- LoRA shape r=16, alpha=32, dropout=0.05 across attention and feed-forward projections, identical to Mini-Assignment 1 and the SFT stage.
- Maximum sequence length 1024, identical to SFT, with truncation if the prompt plus response exceeds this.
- bf16, seed 42, no mid-run checkpoints.

The reference policy is left implicit; with a PEFT-wrapped policy passed to `DPOTrainer`, TRL computes reference log-probabilities by temporarily disabling the active adapter rather than loading a second model copy, which saves model-size in VRAM.


In [16]:
# Section 5.3 - DPO training configuration shared across both beta values.
DPO_BETAS = [0.1, 0.3]

DPO_CFG = {
    "epochs": 5,
    "per_device_batch": 2,
    "grad_accum": 4,                   # effective batch = 8
    "learning_rate": 5e-5,
    "warmup_ratio": 0.03,
    "weight_decay": 0.01,
    "max_length": 1024,
    "lora_r": 16,
    "lora_alpha": 32,
    "lora_dropout": 0.05,
    "lora_target_modules": [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    "seed": SEED,
}
print("DPO config (shared across beta values):")
for k, v in DPO_CFG.items():
    print(f"  {k}: {v}")
print(f"beta values to run: {DPO_BETAS}")


DPO config (shared across beta values):
  epochs: 5
  per_device_batch: 2
  grad_accum: 4
  learning_rate: 5e-05
  warmup_ratio: 0.03
  weight_decay: 0.01
  max_length: 1024
  lora_r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  lora_target_modules: ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj']
  seed: 42
beta values to run: [0.1, 0.3]


## 5.4 Running DPO

The `run_dpo` function below loads the SFT-merged base (from Section 5.1) as the policy, wraps it in a fresh LoRA configured for DPO, and runs TRL's `DPOTrainer` for a given beta value. Each run writes its adapter, log history and a summary file to a tagged output directory: `outputs/ma2-360m-dpo-b01/` for beta 0.1 and `outputs/ma2-360m-dpo-b03/` for beta 0.3.

The trainer's log_history captures the DPO-specific telemetry (reward margins, accuracies, KL to reference) per logging step, which goes into Section 6's discussion of training dynamics.

Each call requires a CUDA GPU; expected wall-clock is roughly 30 to 60 minutes per beta on the RTX 4090 used here.


In [17]:
# Section 5.4 - defines run_dpo(): one DPO run for a given beta value,
# producing a fresh LoRA adapter at outputs/<run_tag>/.
import json
import math
import time
import gc

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, set_seed
from peft import LoraConfig
from trl import DPOConfig, DPOTrainer

# Same path resolution as the cells above.
PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"

def run_dpo(beta: float, run_tag: str):
    set_seed(DPO_CFG["seed"])
    out_dir = PROJECT_ROOT / "outputs" / run_tag
    out_dir.mkdir(parents=True, exist_ok=True)

    # Load the SFT-merged checkpoint as the DPO policy starting point.
    tokenizer = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    model = AutoModelForCausalLM.from_pretrained(
        SFT_MERGED_DIR, torch_dtype=torch.bfloat16
    )

    dpo_lora = LoraConfig(
        r=DPO_CFG["lora_r"],
        lora_alpha=DPO_CFG["lora_alpha"],
        lora_dropout=DPO_CFG["lora_dropout"],
        target_modules=DPO_CFG["lora_target_modules"],
        task_type="CAUSAL_LM",
    )

    args = DPOConfig(
        output_dir=str(out_dir),
        num_train_epochs=DPO_CFG["epochs"],
        per_device_train_batch_size=DPO_CFG["per_device_batch"],
        per_device_eval_batch_size=DPO_CFG["per_device_batch"],
        gradient_accumulation_steps=DPO_CFG["grad_accum"],
        learning_rate=DPO_CFG["learning_rate"],
        warmup_ratio=DPO_CFG["warmup_ratio"],
        weight_decay=DPO_CFG["weight_decay"],
        max_length=DPO_CFG["max_length"],
        bf16=True,
        eval_strategy="steps",
        eval_steps=10,
        logging_steps=5,
        save_strategy="no",
        seed=DPO_CFG["seed"],
        report_to=[],
        beta=beta,
        loss_type="sigmoid",
        remove_unused_columns=False,    # keep prompt/chosen/rejected columns
    )

    trainer = DPOTrainer(
        model=model,
        ref_model=None,                 # implicit via PEFT
        args=args,
        train_dataset=dpo_train,
        eval_dataset=dpo_eval,
        peft_config=dpo_lora,
        processing_class=tokenizer,
    )

    trainable = sum(p.numel() for p in trainer.model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in trainer.model.parameters())

    t0 = time.time()
    trainer.train()
    minutes = (time.time() - t0) / 60
    trainer.save_model(str(out_dir))
    tokenizer.save_pretrained(out_dir)

    final = trainer.evaluate()
    summary = {
        "stage": "dpo",
        "run": run_tag,
        "beta": beta,
        "learning_rate": DPO_CFG["learning_rate"],
        "trainable_params": trainable,
        "total_params": total,
        "epochs": DPO_CFG["epochs"],
        "train_examples": len(dpo_train),
        "val_examples": len(dpo_eval),
        "minutes": round(minutes, 1),
        "final_eval_loss": round(final.get("eval_loss", 0.0), 4),
    }
    # Capture any DPO-specific eval metrics (rewards/accuracies/margins/kl).
    for k, v in final.items():
        if isinstance(v, (int, float)) and k not in summary:
            summary[k] = round(v, 4)

    (out_dir / "log_history.json").write_text(
        json.dumps(trainer.state.log_history, indent=1)
    )
    (out_dir / "summary.json").write_text(json.dumps(summary, indent=1))

    # Free GPU before the next beta run.
    del trainer, model, tokenizer, dpo_lora
    gc.collect()
    torch.cuda.empty_cache()

    print("done:", summary)
    return summary


In [18]:
# Run both beta values back to back. ~30-60 min each on the 4090.
# Both write under outputs/ma2-360m-dpo-b01/ and outputs/ma2-360m-dpo-b03/.
summary_dpo_b01 = run_dpo(0.1, "ma2-360m-dpo-b01")
summary_dpo_b03 = run_dpo(0.3, "ma2-360m-dpo-b03")


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


Adding EOS to train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/225 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/25 [00:00<?, ? examples/s]

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 0}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
10,0.709898,0.650604,1.590115,52101.000000,1.234205,1.066115,0.684892,0.051288,-0.049302,0.576923,0.100590,-312.242366,-276.089174
20,0.703641,0.644340,1.591840,104870.000000,1.236340,1.067344,0.685772,0.010891,-0.126025,0.615385,0.136916,-312.377029,-276.344916
30,0.677233,0.618552,1.589531,151579.000000,1.221298,1.054346,0.683822,0.122639,-0.064730,0.653846,0.187369,-312.004533,-276.140594
40,0.541975,0.590913,1.587160,203072.000000,1.197128,1.028789,0.685108,0.124736,-0.160331,0.769231,0.285067,-311.997540,-276.459266
50,0.455737,0.570098,1.583326,254812.000000,1.166624,1.002975,0.685635,0.144070,-0.221511,0.730769,0.365581,-311.933094,-276.663202
60,0.427439,0.510487,1.579308,303449.000000,1.134073,0.974484,0.683533,0.107186,-0.517052,0.884615,0.624238,-312.056041,-277.648334
70,0.263192,0.488327,1.574023,355928.000000,1.103749,0.943209,0.685220,0.024763,-0.703858,0.769231,0.728622,-312.330782,-278.271024
80,0.260776,0.489108,1.567876,406924.000000,1.063634,0.899495,0.684270,-0.064133,-0.858578,0.846154,0.794445,-312.627103,-278.786755
90,0.151928,0.521957,1.563682,454867.000000,1.036730,0.875214,0.683469,-0.174193,-0.929681,0.846154,0.755488,-312.993970,-279.023761
100,0.136951,0.490398,1.558532,506970.000000,1.000421,0.843787,0.681043,-0.240754,-1.121500,0.846154,0.880747,-313.215841,-279.663165


Training Loss,Validation Loss,Step,Entropy,Num Tokens,Logits/chosen,Logits/rejected,Mean Token Accuracy,Rewards/chosen,Rewards/rejected,Rewards/accuracies,Rewards/margins,Logps/chosen,Logps/rejected
0.056139,0.478564,145,1.545770,731805.000000,0.933360,0.779341,0.679123,-0.459111,-1.477125,0.846154,1.018014,-313.943702,-280.848582


done: {'stage': 'dpo', 'run': 'ma2-360m-dpo-b03', 'beta': 0.3, 'learning_rate': 5e-05, 'trainable_params': 8683520, 'total_params': 370504640, 'epochs': 5, 'train_examples': 225, 'val_examples': 25, 'minutes': 3.8, 'final_eval_loss': 0.4786, 'eval_loss': 0.4786, 'eval_entropy': 1.5458, 'eval_num_tokens': 731805.0, 'eval_logits/chosen': 0.9334, 'eval_logits/rejected': 0.7793, 'eval_mean_token_accuracy': 0.6791, 'eval_rewards/chosen': -0.4591, 'eval_rewards/rejected': -1.4771, 'eval_rewards/accuracies': 0.8462, 'eval_rewards/margins': 1.018, 'eval_logps/chosen': -313.9437, 'eval_logps/rejected': -280.8486}


## 5.5 Sanity check: sample generations

A small set of prompts is run through three model states side by side: the supervised fine-tuned model (the DPO starting point), the DPO-aligned model at beta = 0.1, and the DPO-aligned model at beta = 0.3. Greedy decoding for reproducibility. Expected qualitative differences: the DPO models should produce posting structure closer to the judge's preferred candidates (cleaner section headings, less rambling, fewer truncations), and the higher-beta model should diverge from the SFT model more visibly than the lower-beta model, possibly with regressions visible.

This is qualitative only; the full quantitative comparison (perplexity, LLM-as-judge win-rate) on the 20 frozen prompts is Section 6.


In [19]:
# Section 5.5 - quick sanity check: SFT vs DPO-b01 vs DPO-b03 on a few
# prompts, greedy decoding.
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"

PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

SANITY_PROMPTS = [
    "We are looking for a Senior Data Scientist capable of helping junior members",
    "We need to reforce our team with a DevSecOps Team Lead",
    "Looking for a Functional Tester that can also work with business requirements",
]

def generate(model, tokenizer, prompt: str, max_new_tokens: int = 400) -> str:
    model.eval()
    enc = tokenizer(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tokenizer.eos_token_id,
    )
    return tokenizer.decode(
        out[0][enc["input_ids"].shape[1]:], skip_special_tokens=True
    )

tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
sft  = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16).to("cuda")

base_b01 = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
dpo_b01  = PeftModel.from_pretrained(base_b01, str(DPO_B01_DIR)).to("cuda")

base_b03 = AutoModelForCausalLM.from_pretrained(
    SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
dpo_b03  = PeftModel.from_pretrained(base_b03, str(DPO_B03_DIR)).to("cuda")

for q in SANITY_PROMPTS:
    p = build_prompt(q)
    print("=" * 80)
    print("REQUEST:", q)
    print("=" * 80)
    print("\n--- SFT (DPO starting point) ---")
    print(generate(sft, tok, p))
    print("\n--- DPO beta=0.1 ---")
    print(generate(dpo_b01, tok, p))
    print("\n--- DPO beta=0.3 ---")
    print(generate(dpo_b03, tok, p))
    print()


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

REQUEST: We are looking for a Senior Data Scientist capable of helping junior members

--- SFT (DPO starting point) ---
# Senior Data Scientist

## Summary
We are seeking an experienced Senior Data Scientist to lead the development of complex, high-impact machine learning models. This role requires a blend of deep domain knowledge, strong analytical skills, and the ability to translate complex data into actionable insights. The ideal candidate will be instrumental in driving the development of new, high-impact AI solutions.

## Required Skills
- Deep knowledge of the financial industry and its core concepts
- Expertise in machine learning and deep learning algorithms
- Proficiency in Python and PyTorch
- Experience with data pipelines and data analysis tools
- Ability to translate complex data into actionable insights

## Responsibilities
- Develop and implement high-impact machine learning models
- Lead the development of new, high-impact AI solutions
- Collaborate with teams to trans

## 5.6 Training results and hyperparameter sensitivity

The two DPO runs produced these final metrics on the 25-triple held-out preference set:

| Metric                    | beta = 0.1 | beta = 0.3 |
|---------------------------|-----------:|-----------:|
| `eval_rewards/margins`    |     +0.517 |     +1.018 |
| `eval_rewards/accuracies` |      0.731 |      0.846 |
| `eval_loss`               |      0.539 |      0.479 |
| `eval_mean_token_accuracy`|      0.676 |      0.679 |

Both betas produce a positive margin (chosen ranked above rejected on the held-out set), with accuracies of 73 and 85 percent respectively. The more-constrained beta = 0.3 model outperforms beta = 0.1 on every training-side metric, while both retain the supervised fine-tuned model's token-level accuracy (around 0.68), so the policy did not collapse or lose general fluency while being aligned. The beta = 0.3 result is the headline aligned model in the three-way comparison below; the beta = 0.1 result is retained as the variation point for the sensitivity discussion in Section 7.

### Hyperparameter sensitivity in this setting

These metrics did not emerge on the first attempt. The configuration above (5 epochs, learning rate 5e-5) is the third DPO training iteration we ran on the same data. The earlier two iterations produced essentially-zero or slightly-negative margins, that is, no clear alignment signal: at 1 epoch and learning rate 5e-6 the policy made only 28 optimizer steps and barely moved (margin -0.04, accuracy 0.46); at 5 epochs and the same learning rate the policy made 140 steps but still showed no signal (margin -0.02, accuracy 0.35 to 0.58). The deciding change was the learning rate. 5e-6 is the conventional value cited for full fine-tuning DPO, but it is roughly 10x too low for LoRA-based DPO at this dataset scale, and adjusting it to 5e-5 was a far larger lever than the epoch increase that preceded it.

This is one concrete instance of the well-known sensitivity of alignment training to its hyperparameters, particularly the learning rate and the KL coefficient. The fuller methodological discussion is in Section 7.


# 6. Three-way evaluation

The final comparison contrasts three model states on the same fixed set of prompts: the untrained base, the Mini-Assignment 1 model after supervised fine-tuning, and the aligned model produced in Section 5. Both automatic metrics (perplexity on a held-out set, LLM-as-judge win-rate against a reference) and qualitative side-by-side examples are reported.


## 6.1 Evaluation prompt set

The three-way evaluation contrasts the untrained base, the supervised fine-tuned model (Section 3) and the aligned model (Section 5) on the same fixed set of 20 recruiter queries. Fixing the set up front, before any further training, removes any temptation to cherry-pick favourable prompts and keeps the comparison honest.

The set is split into two ten-prompt sub-sets that test different things:

1. Held-out in-distribution. Ten queries that match the style and topic mix of the supervised fine-tuning data (`data/processed/converted.jsonl`) but are not part of it. These probe how well each model handles the kind of request it has been trained on; they are the fair comparison for in-domain quality.

2. Fresh out-of-distribution. Ten hand-written queries that probe generalisation: unusual roles, niche tech stacks, atypical phrasings and soft-skill emphasis. These probe how well alignment carries over to prompts that look different from the training distribution, and where it may regress.

The two sub-sets are reported separately throughout Section 6 so the report can comment on each setting. The set is held out both from the supervised fine-tuning data and from the preference-generation prompts that Section 4 draws, so it is the only evaluation input the three models share.


In [3]:
# Three-way evaluation prompt set: 20 queries, frozen here, used unchanged
# by every comparison in Section 6. Held out from SFT training data and from
# the preference-generation prompts in Section 4.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

EVAL_PROMPTS = [
    # --- In-distribution: 10 recruiter queries in the style of converted.jsonl ---
    {"id": "ind-01", "subset": "in_distribution",
     "query": "We need a backend engineer to build and maintain Python services on AWS, with hands-on experience in Django and PostgreSQL."},
    {"id": "ind-02", "subset": "in_distribution",
     "query": "Looking for a React developer who can lead frontend architecture on a TypeScript codebase and mentor junior team members."},
    {"id": "ind-03", "subset": "in_distribution",
     "query": "We need an iOS engineer to ship a consumer-facing app in Swift, with experience in SwiftUI and Core Data."},
    {"id": "ind-04", "subset": "in_distribution",
     "query": "Looking for a machine learning engineer to put recommendation models into production, comfortable with PyTorch and Kubernetes."},
    {"id": "ind-05", "subset": "in_distribution",
     "query": "We need a data analyst to build dashboards and explore product data using SQL, dbt and Looker."},
    {"id": "ind-06", "subset": "in_distribution",
     "query": "Looking for a QA automation engineer with Selenium and Playwright experience, to own end-to-end testing of a web platform."},
    {"id": "ind-07", "subset": "in_distribution",
     "query": "We need a cloud engineer to design and operate AWS infrastructure using Terraform, EKS and modern observability tooling."},
    {"id": "ind-08", "subset": "in_distribution",
     "query": "Looking for a full-stack developer comfortable with Node.js, Next.js and PostgreSQL, to ship product features end to end."},
    {"id": "ind-09", "subset": "in_distribution",
     "query": "We need an embedded software engineer to write firmware in C and Rust for low-power IoT devices."},
    {"id": "ind-10", "subset": "in_distribution",
     "query": "Looking for a security engineer to lead application security, threat modelling and code review across multiple product teams."},

    # --- Out-of-distribution: 10 fresh hand-written queries probing generalisation ---
    {"id": "ood-01", "subset": "out_of_distribution",
     "query": "We are hiring a junior developer straight out of a coding bootcamp; the role is heavy on mentorship and pair programming, and the stack is whatever the team is using."},
    {"id": "ood-02", "subset": "out_of_distribution",
     "query": "Looking for a principal staff engineer who has scaled a backend from one to many regions and can act as the org-wide technical authority on distributed systems."},
    {"id": "ood-03", "subset": "out_of_distribution",
     "query": "We need a developer comfortable with Elixir and Phoenix for a real-time messaging product; OTP experience is essential."},
    {"id": "ood-04", "subset": "out_of_distribution",
     "query": "Looking for a quantitative developer to build low-latency trading systems in C++, with a strong background in numerical methods."},
    {"id": "ood-05", "subset": "out_of_distribution",
     "query": "We have a six-month contract for a freelance technical writer who can document a Rust SDK and produce API reference material."},
    {"id": "ood-06", "subset": "out_of_distribution",
     "query": "Looking for someone who is part data engineer and part analytics engineer, comfortable owning the pipeline from ingestion through dbt models to stakeholder dashboards."},
    {"id": "ood-07", "subset": "out_of_distribution",
     "query": "We need a developer who can explain complex topics to non-technical stakeholders, write clear documentation and run customer-facing workshops as part of the role."},
    {"id": "ood-08", "subset": "out_of_distribution",
     "query": "Hiring fully remote across European time zones with an asynchronous-first culture; we need a senior Go engineer to extend our core platform."},
    {"id": "ood-09", "subset": "out_of_distribution",
     "query": "Looking for a game engine programmer comfortable with Unreal Engine 5, gameplay systems and tooling in C++."},
    {"id": "ood-10", "subset": "out_of_distribution",
     "query": "We need a Rust developer to build WebAssembly modules that run client-side in the browser, with experience in performance-critical numerical code."},
]

# Persist as the single source of truth read by the rest of Section 6.
EVAL_DIR  = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_DIR.mkdir(parents=True, exist_ok=True)
EVAL_FILE = EVAL_DIR / "eval_prompts.jsonl"
with open(EVAL_FILE, "w", encoding="utf-8") as f:
    for p in EVAL_PROMPTS:
        f.write(json.dumps(p, ensure_ascii=False) + "\n")

n_ind = sum(1 for p in EVAL_PROMPTS if p["subset"] == "in_distribution")
n_ood = sum(1 for p in EVAL_PROMPTS if p["subset"] == "out_of_distribution")
print(f"Wrote {len(EVAL_PROMPTS)} evaluation prompts to "
      f"{EVAL_FILE.relative_to(PROJECT_ROOT)}")
print(f"  in_distribution:     {n_ind}")
print(f"  out_of_distribution: {n_ood}")


Wrote 20 evaluation prompts to data/processed/ma2/eval_prompts.jsonl
  in_distribution:     10
  out_of_distribution: 10


## 6.2 Generation: producing outputs from each model

For each of the four model states (base SmolLM2-360M, the supervised fine-tuned model, the DPO-aligned model at beta = 0.1, and the DPO-aligned model at beta = 0.3), greedy decoding is run on every one of the 20 evaluation prompts. Greedy decoding rather than sampled decoding is chosen so the four-way comparison is reproducible and reflects each model's most-likely behaviour rather than a sample of its distribution.

The base model has never seen the Section 3.1 prompt template and is not an instruction follower; its generations are expected to look like raw text continuations rather than structured postings. That is the point: the comparison should make explicit what the supervised fine-tuning and DPO stages added beyond pretraining.

Outputs are written per model to `data/processed/ma2/eval_generations/<model>.jsonl`. Each line carries the prompt id, the subset (in_distribution or out_of_distribution), the original query, and the model's generation. Sections 6.4 and 6.5 read these files; the same outputs are used for win-rate judging and for qualitative display.


In [4]:
# Section 6.2 - generate from each of the four models on the 20 eval
# prompts, greedy decoding, max_new_tokens=800. GPU; ~10 min total.
import gc
import json
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR        = PROJECT_ROOT / "data" / "processed" / "ma2"
EVAL_FILE      = MA2_DIR / "eval_prompts.jsonl"
GEN_DIR        = MA2_DIR / "eval_generations"
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"
MA1_LORA_DIR   = PROJECT_ROOT / "outputs" / "mp1-360m" / "lora"

# Base model id from the MA1 adapter config; project convention is to never
# hardcode it.
BASE_MODEL = json.loads((MA1_LORA_DIR / "adapter_config.json").read_text())[
    "base_model_name_or_path"]

GEN_DIR.mkdir(parents=True, exist_ok=True)

# Same template as Section 3.1, defined locally so this cell is
# self-contained.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def build_prompt(query: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n"
    )

def free_gpu():
    gc.collect()
    torch.cuda.empty_cache()

def load_model(tag: str):
    """Return (model, tokenizer) for one of: base | sft | dpo-b01 | dpo-b03."""
    if tag == "base":
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        model = AutoModelForCausalLM.from_pretrained(
            BASE_MODEL, torch_dtype=torch.bfloat16
        )
    elif tag == "sft":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        model = AutoModelForCausalLM.from_pretrained(
            SFT_MERGED_DIR, torch_dtype=torch.bfloat16
        )
    elif tag == "dpo-b01":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        base = AutoModelForCausalLM.from_pretrained(
            SFT_MERGED_DIR, torch_dtype=torch.bfloat16
        )
        model = PeftModel.from_pretrained(base, str(DPO_B01_DIR))
    elif tag == "dpo-b03":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        base = AutoModelForCausalLM.from_pretrained(
            SFT_MERGED_DIR, torch_dtype=torch.bfloat16
        )
        model = PeftModel.from_pretrained(base, str(DPO_B03_DIR))
    else:
        raise ValueError(f"unknown tag {tag!r}")
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return model.to("cuda").eval(), tok

@torch.no_grad()
def gen_one(model, tok, prompt: str, max_new_tokens: int = 800) -> str:
    enc = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **enc, max_new_tokens=max_new_tokens, do_sample=False,
        pad_token_id=tok.eos_token_id,
    )
    return tok.decode(out[0][enc["input_ids"].shape[1]:],
                      skip_special_tokens=True)

# Load the 20 evaluation prompts once.
prompts = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8")]
print(f"Evaluation prompts: {len(prompts)}")

TAGS = ["base", "sft", "dpo-b01", "dpo-b03"]
for tag in TAGS:
    out_path = GEN_DIR / f"{tag}.jsonl"
    print(f"\n=== generating from '{tag}' -> {out_path.name} ===", flush=True)
    t0 = time.time()
    model, tok = load_model(tag)
    with open(out_path, "w", encoding="utf-8") as f:
        for p in prompts:
            text = gen_one(model, tok, build_prompt(p["query"]))
            f.write(json.dumps({
                "prompt_id": p["id"],
                "subset":    p["subset"],
                "query":     p["query"],
                "generation": text,
            }, ensure_ascii=False) + "\n")
    del model, tok
    free_gpu()
    print(f"  done in {(time.time() - t0)/60:.1f} min")


Evaluation prompts: 20

=== generating from 'base' -> base.jsonl ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  done in 5.5 min

=== generating from 'sft' -> sft.jsonl ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  done in 2.0 min

=== generating from 'dpo-b01' -> dpo-b01.jsonl ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  done in 4.7 min

=== generating from 'dpo-b03' -> dpo-b03.jsonl ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  done in 3.9 min


## 6.3 Perplexity on the supervised fine-tuning validation set

Perplexity is the standard automatic metric for a language model. Here we compute it on the supervised fine-tuning validation set (the 250 held-out records times three queries, 750 examples), formatted with the Section 3.1 template. This measures how well each of the four models fits the instruction-following distribution.

The base model is expected to have a substantially higher perplexity than the other three, because it has never seen the prompt template and is not an instruction follower; its generations on the formatted prompt are out of distribution for it. The supervised fine-tuned model should set the lower bound on the SFT-distribution perplexity, and the two DPO models should be close to it (DPO is supposed to shift preferences, not destroy general fluency).

Numbers are stored at `data/processed/ma2/eval_perplexity.json` for Section 6's results discussion.


In [5]:
# Section 6.3 - per-model perplexity on the SFT validation set. GPU.
import gc
import json
import math
import random
import time
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

SFT_DATA       = PROJECT_ROOT / "data" / "processed" / "converted.jsonl"
MA2_DIR        = PROJECT_ROOT / "data" / "processed" / "ma2"
PPL_FILE       = MA2_DIR / "eval_perplexity.json"
SFT_MERGED_DIR = PROJECT_ROOT / "outputs" / "ma2-360m-sft-merged"
DPO_B01_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b01"
DPO_B03_DIR    = PROJECT_ROOT / "outputs" / "ma2-360m-dpo-b03"
MA1_LORA_DIR   = PROJECT_ROOT / "outputs" / "mp1-360m" / "lora"
BASE_MODEL = json.loads((MA1_LORA_DIR / "adapter_config.json").read_text())[
    "base_model_name_or_path"]
SEED = 42

# Reproduce the Section 3.2 90/10 record-level split so the val records are
# exactly the same 250 the SFT model never trained on.
records = [json.loads(l) for l in open(SFT_DATA, encoding="utf-8")]
random.Random(SEED).shuffle(records)
n_val = max(1, len(records) // 10)
val_records = records[:n_val]

# Same template as Section 3.1 so the formatted text matches what SFT
# trained on.
PROMPT_PREAMBLE = (
    "You are a recruitment assistant. Given a brief recruiter request, "
    "write a complete structured job posting in Markdown."
)

def format_example(query: str, jd: str) -> str:
    return (
        f"{PROMPT_PREAMBLE}\n\n"
        f"### Request\n{query}\n\n"
        f"### Posting\n{jd}"
    )

eval_texts = []
for r in val_records:
    for q in r["queries"]:
        eval_texts.append(format_example(q, r["job_description"]))
print(f"perplexity examples: {len(eval_texts)} (from {len(val_records)} val records)")

def load_model(tag: str):
    if tag == "base":
        tok = AutoTokenizer.from_pretrained(BASE_MODEL)
        m   = AutoModelForCausalLM.from_pretrained(BASE_MODEL, torch_dtype=torch.bfloat16)
    elif tag == "sft":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        m   = AutoModelForCausalLM.from_pretrained(SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
    elif tag == "dpo-b01":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        base = AutoModelForCausalLM.from_pretrained(SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
        m   = PeftModel.from_pretrained(base, str(DPO_B01_DIR))
    elif tag == "dpo-b03":
        tok = AutoTokenizer.from_pretrained(SFT_MERGED_DIR)
        base = AutoModelForCausalLM.from_pretrained(SFT_MERGED_DIR, torch_dtype=torch.bfloat16)
        m   = PeftModel.from_pretrained(base, str(DPO_B03_DIR))
    else:
        raise ValueError(tag)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    return m.to("cuda").eval(), tok

@torch.no_grad()
def perplexity(model, tok, texts, max_length: int = 1024) -> float:
    """Token-weighted mean cross-entropy loss across the formatted texts."""
    tot_loss, tot_tok = 0.0, 0
    for txt in texts:
        ids = tok(txt, return_tensors="pt", truncation=True,
                  max_length=max_length).input_ids.to(model.device)
        if ids.shape[1] < 2:
            continue
        out = model(ids, labels=ids)
        n = ids.shape[1] - 1
        tot_loss += out.loss.item() * n
        tot_tok += n
    return math.exp(tot_loss / tot_tok) if tot_tok else float("nan")

results = {}
for tag in ["base", "sft", "dpo-b01", "dpo-b03"]:
    print(f"\n=== perplexity '{tag}' ===", flush=True)
    t0 = time.time()
    model, tok = load_model(tag)
    ppl = perplexity(model, tok, eval_texts)
    print(f"  ppl: {ppl:.2f}  ({(time.time()-t0)/60:.1f} min)")
    results[tag] = round(ppl, 2)
    del model, tok
    gc.collect()
    torch.cuda.empty_cache()

PPL_FILE.write_text(json.dumps({
    "evaluation_set": "sft_val (250 records x 3 queries = 750 examples)",
    "max_length": 1024,
    "perplexity": results,
}, indent=1))
print(f"\nWrote {PPL_FILE.relative_to(PROJECT_ROOT)}")
print(f"summary: {results}")


perplexity examples: 750 (from 250 val records)

=== perplexity 'base' ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  ppl: 11.65  (0.3 min)

=== perplexity 'sft' ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  ppl: 4.54  (0.3 min)

=== perplexity 'dpo-b01' ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  ppl: 4.69  (0.6 min)

=== perplexity 'dpo-b03' ===


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

  ppl: 4.60  (0.6 min)

Wrote data/processed/ma2/eval_perplexity.json
summary: {'base': 11.65, 'sft': 4.54, 'dpo-b01': 4.69, 'dpo-b03': 4.6}


## 6.4 LLM-as-judge win-rate with order-swap control

Pairwise win-rates are computed between every distinct pair of the four model states. With four models there are six pairs (base vs SFT, base vs DPO-b01, base vs DPO-b03, SFT vs DPO-b01, SFT vs DPO-b03, DPO-b01 vs DPO-b03), times 20 evaluation prompts, times two orderings (A-then-B and B-then-A) for position-bias control, gives 240 judge calls. The Gemma-4 e2b judge at concurrency 4 takes roughly fifteen to twenty-five minutes for the full sweep.

The judge uses the same rubric as Section 4.3 (faithfulness, structure, professional and inclusive language, no repetition or truncation), adjusted for pairwise rather than listwise comparison: given a recruiter request and two candidate postings, decide which is better. The judge is asked to end its response with the single line `VERDICT: A` or `VERDICT: B`.

Position-bias control is the central methodological move. Each pair on each prompt is judged twice with the candidates in opposite orders. Only when the two judgements *agree* is the pair-prompt counted as a clean win for one side. When they disagree, the pair-prompt is reported as inconsistent and excluded from the win-rate numerator, but the inconsistency rate itself is reported as a diagnostic for how noisy the judge is.

Outputs go to `data/processed/ma2/eval_winrate.json`. The step is idempotent on the (pair, prompt) key set, so re-running picks up where it left off.


In [6]:
# Section 6.4 - define run_winrate(): pairwise tournament with order-swap
# control, via Gemma-4 on agent_server.
import json
import queue
import re
import threading
import time
import urllib.error
import urllib.request
from itertools import combinations
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MA2_DIR     = PROJECT_ROOT / "data" / "processed" / "ma2"
GEN_DIR     = MA2_DIR / "eval_generations"
EVAL_FILE   = MA2_DIR / "eval_prompts.jsonl"
WINRATE_LOG = MA2_DIR / "winrate_calls.jsonl"   # one row per judge call (resumable)
WINRATE_OUT = MA2_DIR / "eval_winrate.json"     # final aggregated matrix

AGENT_SERVER = "http://localhost:7701"
JUDGE_MODEL  = "gemma-4-e2b-it-q4-kxl-gguf"
ENDPOINT     = f"{AGENT_SERVER}/v1/chat/completions"
TIMEOUT      = 300
WORKERS      = 4

TAGS = ["base", "sft", "dpo-b01", "dpo-b03"]

JUDGE_SYSTEM = (
    "You are an expert recruiter and a careful writing reviewer.\n\n"
    "You will be shown a recruiter request and two candidate job postings, "
    "labelled A and B. Your task is to decide which is better overall.\n\n"
    "Apply the following criteria, in order of importance:\n\n"
    "1. Faithfulness to the request: the posting must reflect the role, "
    "technology stack, seniority and any constraints expressed in the "
    "request.\n"
    "2. Structure and completeness: the posting should include the four "
    "sections '## Summary', '## Required Skills', '## Responsibilities' and "
    "'## Requirements', each with substantive, non-empty content.\n"
    "3. Professional and inclusive language: clear, neutral, free of biased "
    "or discriminatory phrasing.\n"
    "4. Absence of repetition, rambling, or truncation: each section must "
    "read cleanly without filler or mid-sentence stops.\n\n"
    "You may briefly note the strengths and weaknesses of each candidate "
    "before deciding.\n\n"
    "End your response with the single line:\n\n"
    "VERDICT: A\n\n"
    "or:\n\n"
    "VERDICT: B\n\n"
    "Do not write anything after that line."
)

_VERDICT_RE = re.compile(r"VERDICT:\s*([AB])", re.I)

def _build_judge_user(query: str, a: str, b: str) -> str:
    return (
        f"RECRUITER REQUEST:\n{query}\n\n---\n\n"
        f"CANDIDATE A:\n{a}\n\n---\n\n"
        f"CANDIDATE B:\n{b}\n"
    )

def _call_judge(query: str, a: str, b: str) -> str:
    payload = {
        "model": JUDGE_MODEL,
        "messages": [
            {"role": "system", "content": JUDGE_SYSTEM},
            {"role": "user",   "content": _build_judge_user(query, a, b)},
        ],
        "temperature": 0.0,
        "max_tokens": 16384,
    }
    req = urllib.request.Request(
        ENDPOINT, data=json.dumps(payload).encode(),
        headers={"Content-Type": "application/json"},
    )
    with urllib.request.urlopen(req, timeout=TIMEOUT) as r:
        resp = json.load(r)
    return resp["choices"][0]["message"]["content"]

def _parse_verdict(text: str):
    m = _VERDICT_RE.search(text or "")
    if not m:
        return None
    return m.group(1).upper()

def _load_done(path):
    if not path.exists():
        return set()
    out = set()
    for l in open(path, encoding="utf-8"):
        r = json.loads(l)
        out.add((r["pair"], r["prompt_id"], r["order"]))
    return out

def _load_generations():
    by_model = {}
    for tag in TAGS:
        rows = [json.loads(l) for l in open(GEN_DIR / f"{tag}.jsonl", encoding="utf-8")]
        by_model[tag] = {r["prompt_id"]: r["generation"] for r in rows}
    return by_model

def run_winrate():
    by_model = _load_generations()
    prompts  = [json.loads(l) for l in open(EVAL_FILE, encoding="utf-8")]
    pairs    = list(combinations(TAGS, 2))   # 6 unique pairs
    print(f"models: {len(TAGS)}, pairs: {len(pairs)}, prompts: {len(prompts)}")

    # Build full task list.
    tasks = []
    for (a, b) in pairs:
        pair_id = f"{a}_vs_{b}"
        for p in prompts:
            tasks.append((pair_id, a, b, p["id"], p["query"], "ab"))
            tasks.append((pair_id, a, b, p["id"], p["query"], "ba"))

    done = _load_done(WINRATE_LOG)
    todo = [t for t in tasks if (t[0], t[3], t[5]) not in done]
    print(f"calls total {len(tasks)} | already done {len(done)} | todo {len(todo)}")
    if not todo:
        print("nothing to do.")
        return WINRATE_LOG

    q = queue.Queue()
    for t in todo:
        q.put(t)

    f = open(WINRATE_LOG, "a", encoding="utf-8")
    lock = threading.Lock()
    stats = {"ok": 0, "parse_fail": 0, "err": 0}
    t0 = time.time()

    def worker():
        while True:
            try:
                pair_id, a, b, pid, query, order = q.get_nowait()
            except queue.Empty:
                return
            text_a = by_model[a][pid]
            text_b = by_model[b][pid]
            # In 'ba' order, the prompt presents B first as 'A' to the judge,
            # then A as 'B'. The judge's verdict is then re-mapped at parse
            # time below so the verdict is always referenced to (a, b).
            if order == "ab":
                first, second = text_a, text_b
            else:
                first, second = text_b, text_a
            try:
                raw = _call_judge(query, first, second)
            except Exception:
                with lock:
                    stats["err"] += 1
                continue
            v = _parse_verdict(raw)
            with lock:
                if v is None:
                    stats["parse_fail"] += 1
                    continue
                # Map judge's 'A'/'B' (which refer to first/second in this
                # order) back to our canonical (a, b) tags.
                if order == "ab":
                    winner = a if v == "A" else b
                else:
                    winner = b if v == "A" else a
                f.write(json.dumps({
                    "pair":      pair_id,
                    "a":         a, "b": b,
                    "prompt_id": pid, "order": order,
                    "verdict":   v, "winner": winner,
                }, ensure_ascii=False) + "\n")
                f.flush()
                stats["ok"] += 1
                n_seen = stats["ok"] + stats["parse_fail"] + stats["err"]
                if n_seen % 20 == 0:
                    el = time.time() - t0
                    print(f"  ok={stats['ok']} parse_fail={stats['parse_fail']} "
                          f"err={stats['err']} | {el/60:.1f} min")

    threads = [threading.Thread(target=worker, daemon=True) for _ in range(WORKERS)]
    for t in threads: t.start()
    for t in threads: t.join()
    f.close()
    el = time.time() - t0
    print(f"done in {el/60:.1f} min | ok={stats['ok']} "
          f"parse_fail={stats['parse_fail']} err={stats['err']}")

def aggregate_winrate():
    """Read the per-call log and produce the per-pair aggregate matrix."""
    rows = [json.loads(l) for l in open(WINRATE_LOG, encoding="utf-8")]
    # Group by (pair, prompt_id) -> {order -> winner}
    by_key = {}
    for r in rows:
        by_key.setdefault((r["pair"], r["prompt_id"]), {})[r["order"]] = r["winner"]

    # Per-pair stats: consistent_wins[a], consistent_wins[b], inconsistent.
    pair_results = {}
    for (pair, pid), verdicts in by_key.items():
        if len(verdicts) < 2:
            continue
        w_ab = verdicts.get("ab")
        w_ba = verdicts.get("ba")
        # parse pair tag
        a, b = pair.split("_vs_")
        pair_results.setdefault(pair, {"a": a, "b": b, "wins_a": 0,
                                       "wins_b": 0, "inconsistent": 0,
                                       "total": 0})
        d = pair_results[pair]
        d["total"] += 1
        if w_ab == w_ba and w_ab is not None:
            if w_ab == a:
                d["wins_a"] += 1
            elif w_ab == b:
                d["wins_b"] += 1
        else:
            d["inconsistent"] += 1

    # Compute win rates.
    for d in pair_results.values():
        consistent = d["wins_a"] + d["wins_b"]
        d["winrate_a"] = round(d["wins_a"] / consistent, 3) if consistent else None
        d["winrate_b"] = round(d["wins_b"] / consistent, 3) if consistent else None
        d["agreement_rate"] = round(consistent / d["total"], 3) if d["total"] else None

    WINRATE_OUT.write_text(json.dumps(pair_results, indent=1))
    print(f"\nWrote {WINRATE_OUT.relative_to(PROJECT_ROOT)}")
    print("\n--- pair-by-pair win rates ---")
    print(f"  {'pair':24s} {'wins_a':>6s} {'wins_b':>6s} {'incons':>6s} "
          f"{'wr(a)':>6s} {'wr(b)':>6s} {'agree':>6s}")
    for pair, d in pair_results.items():
        wr_a = "n/a" if d["winrate_a"] is None else f"{d['winrate_a']:.2f}"
        wr_b = "n/a" if d["winrate_b"] is None else f"{d['winrate_b']:.2f}"
        ag   = "n/a" if d["agreement_rate"] is None else f"{d['agreement_rate']:.2f}"
        print(f"  {pair:24s} {d['wins_a']:>6d} {d['wins_b']:>6d} "
              f"{d['inconsistent']:>6d} {wr_a:>6s} {wr_b:>6s} {ag:>6s}")


In [7]:
# Run the win-rate tournament. Requires agent_server up on port 7701.
# ~15-25 min for 240 calls at concurrency 4; idempotent on resume.
run_winrate()
aggregate_winrate()


models: 4, pairs: 6, prompts: 20
calls total 240 | already done 0 | todo 240
  ok=20 parse_fail=0 err=0 | 1.1 min
  ok=40 parse_fail=0 err=0 | 2.2 min
  ok=60 parse_fail=0 err=0 | 3.4 min
  ok=80 parse_fail=0 err=0 | 4.6 min
  ok=100 parse_fail=0 err=0 | 5.9 min
  ok=120 parse_fail=0 err=0 | 7.1 min
  ok=140 parse_fail=0 err=0 | 8.4 min
  ok=160 parse_fail=0 err=0 | 9.9 min
  ok=180 parse_fail=0 err=0 | 11.3 min
  ok=200 parse_fail=0 err=0 | 12.8 min
  ok=220 parse_fail=0 err=0 | 14.5 min
  ok=240 parse_fail=0 err=0 | 15.9 min
done in 15.9 min | ok=240 parse_fail=0 err=0

Wrote data/processed/ma2/eval_winrate.json

--- pair-by-pair win rates ---
  pair                     wins_a wins_b incons  wr(a)  wr(b)  agree
  base_vs_sft                   0     19      1   0.00   1.00   0.95
  base_vs_dpo-b01               0     20      0   0.00   1.00   1.00
  base_vs_dpo-b03               0     20      0   0.00   1.00   1.00
  sft_vs_dpo-b01                2     17      1   0.10   0.90   0.95
 

## 6.5 Qualitative side-by-side comparison

The win-rate gives an aggregated picture. To complement it, this subsection displays the four models' generations side by side on a small hand-picked set of prompts, chosen to be diagnostic: two from the in-distribution sub-set and two from the out-of-distribution sub-set. The point of the qualitative pass is to surface behavioural differences that the win-rate compresses away (verbosity, refusals, lost domain knowledge, structural drift, sycophancy), and to give the report concrete textual evidence for the limitations discussion in Section 7.


In [8]:
# Section 6.5 - load the per-model generations and display side-by-side
# for a curated subset of prompts. CPU only.
import json
from pathlib import Path

PROJECT_ROOT = Path.cwd()
while not (PROJECT_ROOT / "data").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

GEN_DIR = PROJECT_ROOT / "data" / "processed" / "ma2" / "eval_generations"
TAGS = ["base", "sft", "dpo-b01", "dpo-b03"]

# Four prompts selected for the qualitative pass: two in-distribution and
# two out-of-distribution. Adjust this list if a different subset reads more
# clearly when the run is done.
QUALITATIVE_IDS = ["ind-01", "ind-04", "ood-01", "ood-03"]

# Load per-model generations.
gens = {}
for tag in TAGS:
    rows = [json.loads(l) for l in open(GEN_DIR / f"{tag}.jsonl", encoding="utf-8")]
    gens[tag] = {r["prompt_id"]: r for r in rows}

for pid in QUALITATIVE_IDS:
    sample = gens["base"].get(pid)
    if sample is None:
        print(f"[missing {pid}]")
        continue
    print("=" * 80)
    print(f"PROMPT ID    : {pid}  ({sample['subset']})")
    print(f"REQUEST      : {sample['query']}")
    print("=" * 80)
    for tag in TAGS:
        print(f"\n--- {tag} ---")
        print(gens[tag][pid]["generation"])
    print()


PROMPT ID    : ind-01  (in_distribution)
REQUEST      : We need a backend engineer to build and maintain Python services on AWS, with hands-on experience in Django and PostgreSQL.

--- base ---

* A brief description of the role
* A brief description of the responsibilities
* A brief description of the skills required
* A brief description of the experience required
* A brief description of the education required
* A brief description of the certifications required
* A brief description of the required skills
* A brief description of the required experience
* A brief description of the required education
* A brief description of the required certifications
* A brief description of the required skills
* A brief description of the required experience
* A brief description of the required education
* A brief description of the required certifications
* A brief description of the required skills
* A brief description of the required experience
* A brief description of the required educatio

# 7. Limitations and critical discussion

Alignment frequently regresses capabilities: the model may become more verbose, refuse more often, lose domain knowledge from earlier stages, or develop sycophancy. This section reports honestly where the alignment helped, where it hurt, and what we would do differently. Honest reporting of these regressions is essential, not optional, since they are the trade-off the technique exposes.

Beyond the model-level regressions, three methodological caveats apply to the way this assignment was carried out, and the report carries them rather than hide them. First, the same Gemma-4 model was used as the supervised fine-tuning data generator, as the preference judge in Section 4, and as the evaluation judge in Section 6 below, because no locally-runnable judge was clearly stronger than Gemma-4 within the 4090's 24 GB envelope; using one model in all three roles introduces self-preference and train-and-evaluate-with-the-same-judge circularity, and that circularity is named explicitly rather than mitigated. Second, the judge exhibits a position bias (candidates presented in position 1 are favoured, position 2 is penalised) and a mild length bias (chosen candidates are about six percent longer on the median than rejected ones), so the preference signal that DPO is trained on inherits these biases. Third, the DPO configuration was iterated to find a working learning rate, as documented in Section 5.6; the first conservative choice (5e-6, conventional for full fine-tuning DPO) produced no alignment signal at all, and the working value (5e-5) is at the high end of the LoRA-DPO range. The amount of compute spent on hyperparameter sensitivity in this project was small; with more compute we would run a small grid over (beta, learning rate) or a Bayesian search on a larger held-out preference set.
